# Module 28 — Exercise 1: FastAPI Request Validation, Pydantic Models, and Endpoints

In this exercise, you will define typed API schemas with Pydantic v2, implement REST endpoints with proper HTTP status codes, and test the endpoints directly using FastAPI's test client.

| Detail | Value |
|---|---|
| **Time** | 35 minutes |
| **Prerequisites** | Module 28 README, Module 17 |



## 1. Defining Pydantic v2 Contracts for APIs


In [ ]:
from pydantic import BaseModel, Field, EmailStr
from typing import Optional

class UserCreate(BaseModel):
    username: str = Field(min_length=3, max_length=20)
    email: str
    age: Optional[int] = Field(default=None, ge=18, le=120)

class UserResponse(BaseModel):
    id: int
    username: str
    email: str
    is_active: bool = True



# Your turn


### Task 1: Create a Task Tracker Router

Define a Pydantic schema `ItemPayload` with `title` (str, min length 1) and `priority` (str, one of `"low"`, `"medium"`, `"high"`).
Implement endpoints:
- `POST /items/` with status code 201 returning created item with generated id.
- `GET /items/{item_id}` returning item or 404 HTTPException if not found.


In [ ]:
# ANSWER 1
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field
from typing import Literal

class ItemPayload(BaseModel):
    title: str = Field(min_length=1)
    priority: Literal["low", "medium", "high"] = "medium"

class ItemOut(ItemPayload):
    id: int

app = FastAPI()
ITEMS_DB: dict[int, ItemOut] = {}
COUNTER = 0

@app.post("/items/", status_code=status.HTTP_201_CREATED, response_model=ItemOut)
def create_item(payload: ItemPayload) -> ItemOut:
    global COUNTER
    COUNTER += 1
    item = ItemOut(id=COUNTER, title=payload.title, priority=payload.priority)
    ITEMS_DB[COUNTER] = item
    return item

@app.get("/items/{item_id}", response_model=ItemOut)
def get_item(item_id: int) -> ItemOut:
    if item_id not in ITEMS_DB:
        raise HTTPException(status_code=404, detail="Item not found")
    return ITEMS_DB[item_id]



## Self-Check Harness


In [ ]:
from fastapi.testclient import TestClient

def check(passed: bool, msg: str) -> bool:
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {msg}")
    return passed

client = TestClient(app)

res_post = client.post("/items/", json={"title": "Fix bug #42", "priority": "high"})
res_get = client.get(f"/items/{res_post.json().get('id', 1)}")
res_missing = client.get("/items/9999")
res_invalid = client.post("/items/", json={"title": "", "priority": "urgent"})

results = [
    check(res_post.status_code == 201, "Task 1: POST returns 201 Created"),
    check(res_get.status_code == 200 and res_get.json()["title"] == "Fix bug #42", "Task 1: GET existing item returns 200 and correct data"),
    check(res_missing.status_code == 404, "Task 1: GET missing item returns 404"),
    check(res_invalid.status_code == 422, "Task 1: Invalid payload triggers 422 Unprocessable Entity"),
]
print(f"Summary: {sum(results)}/{len(results)} checks passed.")

